Retail Streaming Analytics

Notebook: 03_Silver_To_Gold

Purpose:
- Read Silver transactions as stream
- Build Daily Sales Gold aggregate
- Incrementally update Gold table using MERGE

Source:
retailanalytics.silver.silver_transaction

Target:
retailanalytics.gold.daily_sales

In [0]:
from pyspark.sql import functions as F

In [0]:
gold_checkpoint = (
    "/Volumes/retailanalytics/secrets/"
    "kafkacerts/checkpoints/gold_daily_sales"
)

In [0]:
silver_stream = (
    spark.readStream.table(
        "retailanalytics.silver.silver_transaction"
    )
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retailanalytics.gold.daily_sales
(
    sale_date DATE,
    total_revenue DOUBLE,
    total_transactions BIGINT,
    total_units BIGINT
)
USING DELTA;

In [0]:
def upsert_daily_sales(batch_df, batch_id):

    daily_batch = (
        batch_df
        .groupBy(
            F.to_date("transaction_ts").alias("sale_date")
        )
        .agg(
            F.sum("gross_amount").alias("batch_revenue"),
            F.count("*").alias("batch_transactions"),
            F.sum("quantity").alias("batch_units")
        )
    )

    daily_batch.createOrReplaceTempView(
        "daily_sales_updates"
    )

    spark.sql("""
        MERGE INTO retailanalytics.gold.daily_sales AS target
        USING daily_sales_updates AS source
        ON target.sale_date = source.sale_date

        WHEN MATCHED THEN
          UPDATE SET
            target.total_revenue =
                target.total_revenue + source.batch_revenue,

            target.total_transactions =
                target.total_transactions + source.batch_transactions,

            target.total_units =
                target.total_units + source.batch_units

        WHEN NOT MATCHED THEN
          INSERT (
              sale_date,
              total_revenue,
              total_transactions,
              total_units
          )
          VALUES (
              source.sale_date,
              source.batch_revenue,
              source.batch_transactions,
              source.batch_units
          )
    """)

In [0]:
(
    silver_stream.writeStream
        .foreachBatch(upsert_daily_sales)
        .option(
            "checkpointLocation",
            gold_checkpoint
        )
        .trigger(availableNow=True)
        .start()
)